# Previsão de Dados — Regressão Linear Simples

Notebook de **previsão** da colônia Aurora Siger.

Implementa **regressão linear simples** pelo método dos
**mínimos quadrados**, como visto na fase de Ciência de Dados.

A partir de dados históricos (vento × energia gerada), ajustamos
uma reta `y = a*x + b` e a usamos para estimar valores futuros.
Tudo em Python puro, sem NumPy ou bibliotecas de ML.

## 1. Ajuste da reta (mínimos quadrados)

Fórmulas:

- `a = (N*Soma(xy) - Soma(x)*Soma(y)) / (N*Soma(x^2) - Soma(x)^2)`
- `b = (Soma(y) - a*Soma(x)) / N`

In [ ]:
def ajustar_reta(x, y):
    """
    Calcula os coeficientes da reta y = a*x + b pelo metodo
    dos minimos quadrados.

    Retorna:
        (a, b): coeficiente angular e linear da reta.
    """
    n = len(x)
    if n == 0 or n != len(y):
        raise ValueError("Listas x e y devem ter o mesmo tamanho.")

    soma_x = sum(x)
    soma_y = sum(y)
    soma_xy = sum(x[i] * y[i] for i in range(n))
    soma_x2 = sum(x[i] * x[i] for i in range(n))

    denominador = n * soma_x2 - soma_x * soma_x
    if denominador == 0:
        return 0.0, soma_y / n

    a = (n * soma_xy - soma_x * soma_y) / denominador
    b = (soma_y - a * soma_x) / n
    return a, b

## 2. Previsão e qualidade do modelo (R²)

O coeficiente de determinação R² avalia o quão bem a reta
representa os dados (quanto mais perto de 1, melhor).

In [ ]:
def prever(a, b, x_novo):
    """Usa a reta ajustada para estimar y a partir de um novo x."""
    return a * x_novo + b


def r_quadrado(x, y, a, b):
    """
    Calcula o R^2: 1 - (soma dos erros^2) / (variacao total^2).
    """
    n = len(y)
    media_y = sum(y) / n
    soma_erro = sum((y[i] - (a * x[i] + b)) ** 2 for i in range(n))
    soma_total = sum((y[i] - media_y) ** 2 for i in range(n))
    if soma_total == 0:
        return 1.0
    return 1 - (soma_erro / soma_total)

## 3. Função de alto nível

In [ ]:
def prever_energia_eolica(historico_vento, historico_geracao, vento_previsto):
    """
    Preve a energia eolica futura a partir dos historicos.

    Retorna:
        dict com previsao, coeficientes e qualidade do modelo.
    """
    a, b = ajustar_reta(historico_vento, historico_geracao)
    estimativa = prever(a, b, vento_previsto)
    qualidade = r_quadrado(historico_vento, historico_geracao, a, b)
    return {
        "vento_previsto": vento_previsto,
        "energia_estimada": round(estimativa, 1),
        "coef_angular": round(a, 3),
        "coef_linear": round(b, 3),
        "r2": round(qualidade, 3),
    }

## 4. Demonstração

Exemplo do desafio: para vento = 11, a previsão de energia.

In [ ]:
vento = [8, 10, 12, 13, 15]
geracao = [18, 21, 25, 27, 31]

resultado = prever_energia_eolica(vento, geracao, vento_previsto=11)

print("Reta ajustada: energia =", resultado["coef_angular"],
      "* vento +", resultado["coef_linear"])
print("Qualidade (R2):", resultado["r2"])
print("Para vento = 11 -> energia estimada ~", resultado["energia_estimada"])